# 03.01 Why Graphics Needs Cross Product

在 Chapter 02 中，我们已经学习了 dot product。

dot product 的核心作用是衡量两个方向之间的 alignment：

$$
u \cdot v = ||u|| ||v|| \cos(\theta)
$$

如果 u 和 v 都是单位向量，那么 dot product 直接等于两个方向夹角的 cos 值。

因此 dot product 很适合回答这类问题：

- 两个方向是否同向？
- 两个方向是否垂直？
- 表面是否朝向光源？
- 目标是否在相机前方？
- 点在某个方向上的投影长度是多少？

但是 dot product 不能回答另一个图形学中极其重要的问题：

```text
两个方向共同张成的平面，应该朝哪一侧？

## 03.01.01 Dot Product Is Not Enough

假设我们有两个三维向量 u 和 v。

dot product 给出的是一个 scalar：

$$
u \cdot v
$$

这个 scalar 告诉我们 u 和 v 的夹角关系。

但是它不会告诉我们：

- u 和 v 所在平面的法线方向；
- 三角形的正面朝向哪边；
- 顶点顺序是 clockwise 还是 counter-clockwise；
- 相机坐标系中的 right / up / forward 如何互相构造；
- 一个旋转方向是顺时针还是逆时针。

换句话说：

```text
dot product answers: how aligned are two directions?
cross product answers: what direction is perpendicular to both directions?

## 03.01.02 The Core Graphics Problem: Getting a Surface Normal

在图形学中，一个三角形通常由三个点表示：

$$
p_0,\ p_1,\ p_2
$$

为了计算这个三角形的表面法线，我们先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

这两条边都在三角形所在的平面上。

现在问题变成：

```text
如何找到一个同时垂直于 e1 和 e2 的方向？

## 03.01.03 A Concrete Numerical Example

考虑一个位于 xy 平面上的三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

构造两条边：

$$
e_1=p_1-p_0=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=p_2-p_0=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

根据右手坐标系约定：

$$
e_1 \times e_2=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

所以这个三角形的 normal 指向正 z 方向。

如果交换顺序：

$$
e_2 \times e_1=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

方向会完全反过来。

这说明 cross product 不满足交换律：

$$
u \times v=-(v \times u)
$$

## 03.01.04 Why Direction Matters in Graphics

cross product 的方向不是一个纯数学细节，而是直接影响图形学结果。

如果三角形 normal 方向错了，会影响：

1. lighting  
   表面可能被错误地认为背对光源。

2. back-face culling  
   本来应该显示的三角形可能被剔除。

3. shading  
   法线方向错误会导致明暗反转。

4. collision detection  
   接触面方向错误会导致响应方向错误。

5. camera basis  
   right、up、forward 构造顺序错误会导致相机坐标系翻转。

因此 cross product 的方向约定必须和坐标系约定一起理解。

## 03.01.05 Cross Product as Orientation Information

dot product 的结果是 scalar。

cross product 的结果是 vector。

这两者的几何意义不同：

```text
dot product:
two vectors → scalar
measures alignment

cross product:
two 3D vectors → vector
produces a perpendicular direction

## 03.01.06 Graphics Applications Preview

后续章节中，cross product 会反复出现。

第一类应用是 surface normal。

给定三角形三个点：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

$$
n=\operatorname{normalize}(e_1 \times e_2)
$$

第二类应用是 camera basis。

如果我们知道 camera forward 和 world up，可以构造 camera right：

$$
r=\operatorname{normalize}(f \times up)
$$

然后再构造真正的 camera up：

$$
u=\operatorname{normalize}(r \times f)
$$

第三类应用是 back-face culling。

如果 triangle normal 和 view direction 的关系表明三角形背对相机，那么可以不绘制它。

第四类应用是面积。

cross product 的长度和两个向量张成的平行四边形面积有关：

$$
||u \times v||=||u|| ||v|| \sin(\theta)
$$

三角形面积则是它的一半：

$$
A=\frac{1}{2}||e_1 \times e_2||
$$

## 03.01.07 NumPy Example

下面先用 NumPy 验证最基本的 cross product 方向。

我们使用：

```text
x × y = z
y × x = -z

In [ ]:
import numpy as np

x = np.array([1.0, 0.0, 0.0])
y = np.array([0.0, 1.0, 0.0])
z = np.array([0.0, 0.0, 1.0])

cross_xy = np.cross(x, y)
cross_yx = np.cross(y, x)

print("x cross y:", cross_xy)
print("y cross x:", cross_yx)
print("Expected z:", z)
print("Expected -z:", -z)

x cross y: [0. 0. 1.]
y cross x: [ 0.  0. -1.]
Expected z: [0. 0. 1.]
Expected -z: [-0. -0. -1.]


## 03.01.08 NumPy Example: Triangle Normal

现在考虑三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

先构造边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

再计算：

$$
n=e_1 \times e_2
$$

In [2]:
def normalize(v):
    """归一化向量；zero vector 不能归一化。"""
    v = np.asarray(v, dtype=float)
    length = np.linalg.norm(v)

    if np.isclose(length, 0.0):
        raise ValueError("Cannot normalize a zero vector.")

    return v / length


p0 = np.array([0.0, 0.0, 0.0])
p1 = np.array([1.0, 0.0, 0.0])
p2 = np.array([0.0, 1.0, 0.0])

e1 = p1 - p0
e2 = p2 - p0

n = np.cross(e1, e2)
n_hat = normalize(n)

print("e1:", e1)
print("e2:", e2)
print("normal:", n)
print("unit normal:", n_hat)

e1: [1. 0. 0.]
e2: [0. 1. 0.]
normal: [0. 0. 1.]
unit normal: [0. 0. 1.]


## 03.01.09 What Happens If We Reverse Vertex Order?

如果交换 p1 和 p2，相当于改变三角形的 winding order。

原来是：

$$
n=(p_1-p_0) \times (p_2-p_0)
$$

交换后是：

$$
n'=(p_2-p_0) \times (p_1-p_0)
$$

因为 cross product 反交换：

$$
n'=-n
$$

所以 normal direction 会翻转。

这就是 winding order 会影响 triangle normal 的原因。

## 03.01.11 Exercises

### Exercise 1

给定：

$$
u=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

请判断：

$$
u \times v
$$

的方向。

再判断：

$$
v \times u
$$

的方向。

---

### Exercise 2

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
0 \\
3 \\
0
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

并判断：

$$
e_1 \times e_2
$$

的方向。

---

### Exercise 3

如果一个三角形的顶点顺序从：

$$
p0 → p1 → p2
$$

改成：
$$
p0 → p2 → p1
$$
请说明 normal direction 会发生什么变化。

# 03.02 Cross Product Definition

上一节我们已经知道，cross product 用来从两个 3D 向量构造一个新的方向。

这个新方向有两个关键性质：

1. 它垂直于第一个输入向量；
2. 它垂直于第二个输入向量。

也就是说，如果输入是 u 和 v，那么 cross product 输出的是：

$$
u \times v
$$

这个结果不是 scalar，而是一个 vector。

这和 dot product 完全不同：

```text
dot product:
u · v → scalar

cross product:
u × v → vector

## 03.02.01 Cross Product Only Makes Sense Here as a 3D Operation

在本课程的图形学语境中，我们主要讨论 3D cross product。

给定两个三维向量：

$$
u=
\begin{bmatrix}
u_x \\
u_y \\
u_z
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
v_x \\
v_y \\
v_z
\end{bmatrix}
$$

它们的 cross product 仍然是一个三维向量：

$$
u \times v=
\begin{bmatrix}
? \\
? \\
?
\end{bmatrix}
$$

这个输出向量的方向由右手定则决定。

它的长度和 u、v 张成的面积有关。

这一节先重点掌握代数公式。

## 03.02.02 Algebraic Definition

给定：

$$
u=
\begin{bmatrix}
u_x \\
u_y \\
u_z
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
v_x \\
v_y \\
v_z
\end{bmatrix}
$$

cross product 定义为：

$$
u \times v=
\begin{bmatrix}
u_y v_z-u_z v_y \\
u_z v_x-u_x v_z \\
u_x v_y-u_y v_x
\end{bmatrix}=
\begin{bmatrix}
D_{yz} \\
D_{zx} \\
D_{xy}
\end{bmatrix}
$$

也就是说：

$$
(u \times v)_x=u_y v_z-u_z v_y
$$

$$
(u \times v)_y=u_z v_x-u_x v_z
$$

$$
(u \times v)_z=u_x v_y-u_y v_x
$$

注意第二个分量很容易写错。

它是：

$$
u_z v_x-u_x v_z
$$

不是：

$$
u_x v_z-u_z v_x
$$

## 03.02.03 Concrete Example: x Cross y

令：

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

根据公式：

$$
x \times y=
\begin{bmatrix}
0 \cdot 0-0 \cdot 1 \\
0 \cdot 0-1 \cdot 0 \\
1 \cdot 1-0 \cdot 0
\end{bmatrix}
$$

所以：

$$
x \times y=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

这正好是 z 方向。

因此在右手坐标系中：

$$
x \times y=z
$$

## 03.02.04 Reversing the Order

现在交换顺序：

$$
y \times x
$$

其中：

$$
y=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
x=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

根据公式：

$$
y \times x=
\begin{bmatrix}
1 \cdot 0-0 \cdot 0 \\
0 \cdot 1-0 \cdot 0 \\
0 \cdot 0-1 \cdot 1
\end{bmatrix}
$$

所以：

$$
y \times x=
\begin{bmatrix}
0 \\
0 \\
-1
\end{bmatrix}
$$

也就是说：

$$
y \times x=-z
$$

因此：

$$
u \times v=-(v \times u)
$$

cross product 不满足交换律。

## 03.02.05 Cross Product Is Orthogonal to Both Inputs

cross product 的核心性质是：

$$
u \times v \perp u
$$

同时：

$$
u \times v \perp v
$$

换成 dot product 表示就是：

$$
(u \times v) \cdot u=0
$$

$$
(u \times v) \cdot v=0
$$

这就是为什么 cross product 可以用来计算 surface normal。

如果三角形的两条边是：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

那么：

$$
n=e_1 \times e_2
$$

会同时垂直于 e1 和 e2。

因此 n 垂直于三角形所在平面。

## 03.02.06 Numerical Example with Non-Axis Vectors

令：

$$
u=
\begin{bmatrix}
1 \\
2 \\
3
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
4 \\
5 \\
6
\end{bmatrix}
$$

根据公式：

$$
u \times v=
\begin{bmatrix}
2 \cdot 6-3 \cdot 5 \\
3 \cdot 4-1 \cdot 6 \\
1 \cdot 5-2 \cdot 4
\end{bmatrix}
$$

所以：

$$
u \times v=
\begin{bmatrix}
-3 \\
6 \\
-3
\end{bmatrix}
$$

验证它和 u 垂直：

$$
(u \times v) \cdot u=(-3)\cdot 1+6\cdot 2+(-3)\cdot 3
$$

$$
(u \times v) \cdot u=0
$$

验证它和 v 垂直：

$$
(u \times v) \cdot v=(-3)\cdot 4+6\cdot 5+(-3)\cdot 6
$$

$$
(u \times v) \cdot v=0
$$

## 03.02.07 NumPy Implementation: Using np.cross

NumPy 已经提供了 cross product 函数：

```text
np.cross(u, v)

In [3]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])

cross_uv = np.cross(u, v)
cross_vu = np.cross(v, u)

print("u cross v:", cross_uv)
print("v cross u:", cross_vu)
print("-(v cross u):", -cross_vu)

print("Are they opposite?", np.allclose(cross_uv, -cross_vu))

u cross v: [-3.  6. -3.]
v cross u: [ 3. -6.  3.]
-(v cross u): [-3.  6. -3.]
Are they opposite? True


## 03.02.10 Graphics Connection: Triangle Normal

cross product 最直接的图形学应用是计算三角形法线。

给定三角形三个点：

$$
p_0,\ p_1,\ p_2
$$

先构造两条边：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

然后：

$$
n=e_1 \times e_2
$$

最后归一化：

$$
\hat{n}=\frac{n}{||n||}
$$

这里必须注意：

```text
e1 × e2 和 e2 × e1 的方向相反。

## 03.02.11 Degenerate Triangle Case

如果三角形的三个点共线，那么两条边不能张成一个真正的平面。

例如：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

此时：

$$
e_1=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
e_2=
\begin{bmatrix}
2 \\
0 \\
0
\end{bmatrix}
$$

e1 和 e2 共线。

所以：

$$
e_1 \times e_2=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

zero vector 不能 normalized。这类三角形称为 degenerate triangle，在图形学中，它没有有效的 surface normal。

## 03.02.14 Exercises

### Exercise 1

给定：

$$
u=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
0 \\
0 \\
1
\end{bmatrix}
$$

请计算：

$$
u \times v
$$

并判断结果指向哪个坐标轴方向。

---

### Exercise 2

给定：

$$
u=
\begin{bmatrix}
1 \\
2 \\
0
\end{bmatrix}
$$

$$
v=
\begin{bmatrix}
3 \\
1 \\
0
\end{bmatrix}
$$

请手动计算：

$$
u \times v
$$

并说明为什么结果只在 z 方向上有分量。

---

### Exercise 3

给定三角形：

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
0 \\
1 \\
0
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
1 \\
0 \\
0
\end{bmatrix}
$$

请计算：

$$
e_1=p_1-p_0
$$

$$
e_2=p_2-p_0
$$

以及：

$$
n=e_1 \times e_2
$$

判断这个 normal 指向正 z 方向还是负 z 方向。

---

### Exercise 4

为什么下面三个点不能定义出一个有效的三角形 normal？

$$
p_0=
\begin{bmatrix}
0 \\
0 \\
0
\end{bmatrix}
$$

$$
p_1=
\begin{bmatrix}
1 \\
1 \\
1
\end{bmatrix}
$$

$$
p_2=
\begin{bmatrix}
2 \\
2 \\
2
\end{bmatrix}
$$

请从 cross product 的角度解释。

---

### Exercise 5

用 NumPy 实现一个函数：

```text
triangle_normal(p0, p1, p2)